# PersonSense — finding the best on-device VLM for person bbox detection

## Goal

Run a vision–language model **on a phone CPU only** to detect people and return tight
bounding boxes, fast enough to feel interactive.

## Starting point — what we already knew

A colleague had already verified two things on the Snapdragon 8 Elite (S25 Ultra):

1. **`-t 4` (four threads) with no CPU-mask affinity** gives the best generation
   throughput in `llama-bench`. Pinning to specific cores hurts because the Linux
   scheduler on S25 places threads on the 2 prime + 2 perf cores when load is high.
2. **Flash-attention + `Q8_0` KV cache** is a free win on CPU for memory bandwidth.

Those flags are constant across every experiment in this notebook, so the analysis
below isolates the choices that *actually move the operating point*.

## What this notebook answers

- Which model family — **Qwen3.5 (Mamba-hybrid) or Qwen3-VL (vanilla transformer)**?
- Which size — 0.8B or 2B?
- Which weight quant — Q4_0 or Q8_0 for the LM? F16 or Q8_0 for the mmproj?
- How many visual tokens per image?
- Where is the **accuracy cliff** — i.e. how aggressively can we cap visual tokens before mAP collapses?
- What is the final on-device operating point, and how fast is the app that ships it?

## Setup

**Hardware.** Two phones: Samsung Galaxy S25 Ultra (Snapdragon 8 Elite, Adreno 830,
12 GB) and Fairphone 5 (Snapdragon QCM6490, Adreno 643, 8 GB). Comparison phones for
the same benchmark but different SoC class.

**Runtime.** [`llama.cpp`](https://github.com/ggml-org/llama.cpp) (March 2026
snapshot) with `mtmd` for multimodal. Two builds: the Qualcomm pkg-snapdragon
build on S25 (OpenCL libs available but we use CPU-only with `-ngl 0
--no-mmproj-offload`) and a CPU-only NDK build on FP5.

**Workload.** 50 single-person images from COCO val2017 with ground-truth bboxes.
Each run produces a JSON-formatted bbox via the same ChatML prompt with an empty
`<think>` block to disable Qwen's thinking phase. Output is parsed and compared
against GT with standard IoU; the headline metric is **mAP@.5 = fraction of
images with IoU ≥ 0.5**.

**Metrics.**
- **TTFT** (time to first token) = image encode + image-token decode + text
  prefill + 1-token gen. This is what the user feels as latency.
- **Generation tok/s** during the autoregressive decode.
- **mAP@.5** for accuracy.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

HERE = Path('.').resolve()
df = pd.read_csv(HERE / 'personsense-bench-results.csv')

# Compute TTFT in seconds and a 'total' (TTFT + gen time) for one bbox response.
df['ttft_s'] = df['ttft_ms'] / 1000.0
df['gen_s']  = df['gen_ms'] / 1000.0
df['total_s'] = df['ttft_s'] + df['gen_s']

# Drop incomplete rows (timing missing).
df = df.dropna(subset=['ttft_s'])

print(f'{len(df):,} runs in CSV')
print(f'Devices: {sorted(df.device.unique().tolist())}')
print(f'Families: {sorted(df.family.unique().tolist())}')
print(f'Sizes: {sorted(df.model.unique().tolist())}')

## Experiment matrix

All runs combined across both phones and all sweeps:

In [ ]:
def summary_table(d):
    g = (d.groupby(['device','family','model','quant','mmproj','bound','tok'])
         .agg(n=('iou','size'),
              mIoU=('iou','mean'),
              mAP50=('iou', lambda s: (s >= 0.5).mean()),
              TTFT_s=('ttft_s','mean'),
              gen_tok_s=('tok_per_sec','mean'))
         .round({'mIoU':2,'mAP50':2,'TTFT_s':1,'gen_tok_s':1}))
    return g.reset_index()

summary_table(df).head(20)

## Finding 1 — Architecture beats size

Compare Qwen3.5 2B (Mamba-hybrid) vs Qwen3-VL 2B (vanilla transformer trained with
vision) at matched configs on S25. Same parameter count, same `image-min-tokens 256`,
same quant. The **only** difference is the model family.

In [ ]:
head_to_head = (
    df.query("device=='s25' and bound=='min' and model=='2B' and mmproj=='F16'")
      .groupby(['family','quant','tok']).agg(
          TTFT=('ttft_s','mean'), tok_s=('tok_per_sec','mean'),
          mAP=('iou', lambda s: (s>=0.5).mean()))
      .round(2)
      .reset_index()
      .pivot_table(index=['quant','tok'], columns='family',
                   values=['TTFT','tok_s','mAP'])
)
head_to_head

**Takeaway.** Qwen3-VL wins on mAP at every single matched config (+0.04 to +0.06).
Speed is a wash at tok256 and Qwen3.5 pulls ahead at tok576/1024 because Mamba
recurrence doesn't scale with KV cache size while Qwen3-VL's vanilla attention does.

But the accuracy gap is structural — small Mamba-hybrid models are weak at
**grounding** tasks (bbox prediction needs precise spatial reasoning), which is
exactly the regime Qwen3-VL was trained for. For our task, **Qwen3-VL is the right family**.

We also tested Qwen3.5 **0.8B** (the small Mamba-hybrid) as a "maybe small is fast enough"
fallback. It tops out at mAP 0.50 — below Qwen3-VL 2B's plateau of 0.60. Skip it for
bbox tasks; reserve it for cheap caption/classification.

## Finding 2 — Q8_0 LM + Q8_0 mmproj is the sweet spot

Once we picked Qwen3-VL 2B, the next two knobs are LM quant and mmproj quant.

On S25 with TTFT-sensitive workloads:

- **Q8_0 LM** has higher mAP than Q4_0 (more weight precision); on a 2B model the
  i8mm matmul kernels make Q8 about as fast as Q4 despite 2× the bytes.
- **Q8_0 mmproj** ≈ 50 % the size of F16 mmproj and ~3× faster to encode an image,
  with negligible accuracy cost when paired with Q8_0 LM.

In [ ]:
knobs = (df.query("device=='s25' and family=='q3vl' and bound=='min' and tok==256")
         .groupby(['quant','mmproj']).agg(
             TTFT_s=('ttft_s','mean'),
             gen_tok_s=('tok_per_sec','mean'),
             mAP=('iou', lambda s: (s>=0.5).mean()),
         ).round({'TTFT_s':1, 'gen_tok_s':1, 'mAP':2}))
knobs

**Q8_0 mmproj cuts TTFT from 21.8 s → 14.2 s for the same accuracy** when the LM
is also Q8_0. That's the biggest single TTFT lever we found before touching visual
tokens.

## Finding 3 — The accuracy cliff is at 72 visual tokens

Qwen3-VL was trained on high-resolution images. The model file even prints a
runtime warning recommending **at least 1024 visual tokens** for grounding tasks
([llama.cpp#16842](https://github.com/ggml-org/llama.cpp/issues/16842)).

Does mAP actually collapse below that? We ran a fine-grained max-tokens sweep with
the recommended LM/mmproj config (Q8_0/Q8_0) on S25:

- max-tokens caps in {32, 64, **72, 80, 88,** 96, 128, 196, 256}
- 50 COCO images per cap
- pure CPU

In [ ]:
cliff = (df.query("device=='s25' and family=='q3vl' and bound=='max' "
                  "and quant=='Q8_0' and mmproj=='Q8_0'")
         .groupby('tok').agg(
             n=('iou','size'),
             TTFT_s=('ttft_s','mean'),
             mAP=('iou', lambda s: (s>=0.5).mean())
         ).round({'TTFT_s':2,'mAP':2})
         .reset_index())
cliff

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(cliff['tok'], cliff['mAP'], marker='o', linewidth=3, markersize=12, color='#4477AA')
for x,y in zip(cliff['tok'], cliff['mAP']):
    ax1.annotate(f'{y:.2f}', (x,y), textcoords='offset points', xytext=(0,10), ha='center', fontweight='bold')
ax1.axhline(0.60, color='#117733', linestyle='--', alpha=.6, label='full-quality plateau')
ax1.axvspan(64, 72, alpha=0.2, color='orange', label='cliff (64→72)')
ax1.set_xlabel('--image-max-tokens'); ax1.set_ylabel('mAP@.5')
ax1.set_title('S25 — accuracy cliff'); ax1.grid(alpha=.3); ax1.legend()

ax2.plot(cliff['tok'], cliff['TTFT_s'], marker='o', linewidth=3, markersize=12, color='#332288')
for x,y in zip(cliff['tok'], cliff['TTFT_s']):
    ax2.annotate(f'{y:.1f}s', (x,y), textcoords='offset points', xytext=(0,10), ha='center', fontweight='bold')
ax2.set_xlabel('--image-max-tokens'); ax2.set_ylabel('TTFT (seconds)')
ax2.set_title('S25 — TTFT cost'); ax2.grid(alpha=.3)
fig.tight_layout()
plt.show()

**The cliff is between 64 and 72 visual tokens.** mAP jumps from 0.36 → 0.48
across an 8-token gap. From 72 onward it climbs slowly (0.48 → 0.52 by max=88)
and plateaus near 0.56 around max=196. The runtime warning's "1024 tokens" is
conservative — for our person-bbox task **72 is enough.**

Why 72? At Qwen3-VL's 32×32-pixel-per-token tiling, max=72 produces roughly a
10×7 grid (~320×224 px). A typical COCO person is ~120 px tall — so ~4 tiles
vertically. Below that the model can detect "a person exists" but can't draw a
tight box.

## Finding 4 — FP5 confirms the cliff is a model property

We re-ran the max-tokens sweep on the Fairphone 5 with the same Q3VL 2B Q8/Q8
config. The mAP curve is essentially identical (same model, same prompt, same
images), but everything is ~5× slower in seconds because of the weaker SoC.

In [ ]:
both = (df.query("family=='q3vl' and bound=='max' "
                 "and quant=='Q8_0' and mmproj=='Q8_0'")
        .groupby(['device','tok']).agg(
             TTFT_s=('ttft_s','mean'),
             mAP=('iou', lambda s: (s>=0.5).mean())
        ).round({'TTFT_s':1,'mAP':2}))
both

## Finding 5 — Wall-clock decomposition

Where does the 2.8 s of TTFT at max=72 on S25 actually go?

In [ ]:
decomp = (df.query("device=='s25' and family=='q3vl' and bound=='max' "
                   "and quant=='Q8_0' and mmproj=='Q8_0' and tok==72")
          .agg(encode_s=('encode_ms','mean'),
               decode_s=('decode_ms','mean'),
               prompt_eval_s=('prompt_eval_ms','mean'),
               gen_s=('gen_ms','mean')) / 1000.0).round(2)
decomp

Encode + decode + prefill (= TTFT minus the first-token gen) accounts for ~99 % of
wall-clock; the autoregressive decode of ~36 bbox-JSON tokens is barely visible. So
the levers that matter are **anything that shrinks encode or prefill**, not the LM's
raw gen tok/s.

## The operating point — "CVPR2026"

Putting the four findings together:

| knob | choice | reason |
|---|---|---|
| model family | **Qwen3-VL** | wins mAP at every matched config vs Qwen3.5 |
| model size | **2B** | 0.8B caps at mAP 0.50 vs 2B's 0.60 plateau |
| LM quant | **Q8_0** | +2–6pp mAP over Q4_0 at no real CPU speed cost |
| mmproj quant | **Q8_0** | ~3× faster encode at no real mAP cost |
| `--image-max-tokens` | **72** | cliff edge; +50 visual tokens vs the cliff bottom but full prefix cost still tiny |
| backend | **CPU only** | apples-to-apples baseline; OpenCL is the next paper |

**Measured outcome on S25:** TTFT **2.8 s**, mAP@.5 **0.48**, gen tok/s 28.6.

That's the **PersonSense CVPR** APK we shipped.
